# Text-based decomposition of CLIP's image representation

CLIP's image embedding is a sum. The class token's final residual state is the patch embedding
plus what every attention head and every MLP wrote into it, and the final layer norm and the
projection into the shared image-text space are linear once the norm's scale is fixed. So the
embedding splits exactly into one term per head and per MLP, and each term lives in the same
space as CLIP's text embeddings, where sentences can describe it. This recipe reproduces the
paper's three steps on CLIP ViT-B/32:

1. split the image embedding into per-head and per-MLP terms, and check that they add up;
2. mean-ablate groups of terms and watch zero-shot accuracy: the last attention layers carry it,
   the MLPs barely matter;
3. run TextSpan, which describes each head's term by the sentences that explain most of its
   variance across pictures, and find heads for colour, shape and number.

Paper: Yossi Gandelsman, Alexei A. Efros and Jacob Steinhardt, 2024, "Interpreting CLIP's Image
Representation via Text-Based Decomposition", ICLR 2024, https://arxiv.org/abs/2310.05916.

TorchLens supplies the terms: the `result` facet of every `CLIPAttention` module is each head's
contribution to the residual stream (`[picture, token, head, width]`), read from the fused
attention kernel's saved inputs under `reconstruction_ready=True`, and every module's output is
a facet too. The model is the public `openai/clip-vit-base-patch32` checkpoint (about 600 MB,
downloaded once from the Hugging Face Hub); the recipe runs on a laptop CPU in a few minutes.
The paper used ImageNet and a pool of about 3,500 sentences; here 24 drawn pictures (three
shapes, four colours, one or three copies) and a pool of 39 sentences keep it small.

In [ ]:
# recipe-test: tool=clip expected=(24, 3, 224, 224)
import torch
import transformers
from PIL import Image, ImageDraw

import torchlens as tl

MODEL_ID = "openai/clip-vit-base-patch32"
REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"  # pinned so the numbers below reproduce
model = transformers.CLIPModel.from_pretrained(MODEL_ID, revision=REVISION).eval()
processor = transformers.CLIPProcessor.from_pretrained(MODEL_ID, revision=REVISION)

RGB = {
    "red": (220, 30, 30),
    "green": (30, 170, 50),
    "blue": (30, 60, 220),
    "yellow": (240, 210, 20),
}
SHAPES = ("circle", "square", "triangle")


def draw_scene(shape: str, colour: str, copies: int, size: int = 224) -> Image.Image:
    """Draw ``copies`` filled shapes of one colour in a row on a plain background."""

    picture = Image.new("RGB", (size, size), (245, 245, 245))
    pen = ImageDraw.Draw(picture)
    radius = 60 if copies == 1 else 30
    for index in range(copies):
        x, y = size * (index + 1) // (copies + 1), size // 2
        box = (x - radius, y - radius, x + radius, y + radius)
        if shape == "circle":
            pen.ellipse(box, fill=RGB[colour])
        elif shape == "square":
            pen.rectangle(box, fill=RGB[colour])
        else:
            pen.polygon(
                [(x, y - radius), (x - radius, y + radius), (x + radius, y + radius)],
                fill=RGB[colour],
            )
    return picture


scenes = [(shape, colour, copies) for shape in SHAPES for colour in RGB for copies in (1, 3)]
pixel_values = processor(images=[draw_scene(*scene) for scene in scenes], return_tensors="pt")[
    "pixel_values"
]


def embed_text(sentences: list[str]) -> torch.Tensor:
    """Return CLIP's unit-length text embeddings for ``sentences``."""

    tokens = processor(text=sentences, return_tensors="pt", padding=True)
    with torch.no_grad():
        features = model.get_text_features(**tokens)
    features = features if torch.is_tensor(features) else features.pooler_output
    return features / features.norm(dim=-1, keepdim=True)


RESULT = tuple(pixel_values.shape)

## 1. Split the image embedding into per-head and per-MLP terms

Each residual layer adds `self_attn(layer_norm1(x))` and then `mlp(layer_norm2(x))` to the
stream, so the class token's final state is `pre_layrnorm`'s output plus, for every layer, the
heads' `result` terms, the output projection's bias, and the MLP's output. The final
`post_layernorm` subtracts the state's mean and divides by its standard deviation; with that
standard deviation fixed per picture, each term passes through the norm's scale and
`visual_projection` on its own, and the norm's bias becomes one more term.

In [ ]:
# recipe-test: tool=clip expected=True
trace = tl.trace(
    model.vision_model,
    pixel_values,
    capture=tl.options.CaptureOptions(reconstruction_ready=True),
)
tower = model.vision_model
terms: dict[str, torch.Tensor] = {}
with torch.no_grad():
    terms["embeddings"] = trace.modules["pre_layrnorm"].facets["out"].value[:, 0].clone()
    for layer, block in enumerate(tower.encoder.layers):
        heads = trace.modules[f"encoder.layers.{layer}.self_attn"].facets["result"].value
        heads = heads[:, 0].clone()  # the class token's row: [picture, head, width]
        for head in range(heads.shape[1]):
            terms[f"L{layer}.H{head}"] = heads[:, head]
        terms[f"L{layer}.attn_bias"] = block.self_attn.out_proj.bias.expand_as(heads[:, 0])
        terms[f"L{layer}.mlp"] = (
            trace.modules[f"encoder.layers.{layer}.mlp"].facets["out"].value[:, 0].clone()
        )

    final_state = sum(terms.values())
    norm = tower.post_layernorm
    scale = (final_state.var(-1, keepdim=True, unbiased=False) + norm.eps).sqrt()
    projection = model.visual_projection.weight
    parts = {
        name: ((term - term.mean(-1, keepdim=True)) / scale * norm.weight) @ projection.T
        for name, term in terms.items()
    }
    parts["post_layernorm.bias"] = (norm.bias @ projection.T).expand_as(parts["embeddings"])
    image_embeds = model.get_image_features(pixel_values=pixel_values)
    image_embeds = image_embeds if torch.is_tensor(image_embeds) else image_embeds.pooler_output

reassembled = sum(parts.values())
gap = (reassembled - image_embeds).abs().max().item()
print(f"{len(parts)} terms; largest gap to CLIP's own image embedding: {gap:.1e}")
RESULT = gap < 1e-3

## 2. Mean-ablate groups of terms

Replace a group's terms by their mean over the pictures and score zero-shot classification
into the twelve "a photo of a red circle" labels. The paper found that the MLPs and the early
attention layers can be mean-ablated with little loss, while the last four attention layers
cannot. The same holds here.

In [ ]:
# recipe-test: tool=clip expected=(True, True, True)
labels = [f"a photo of a {colour} {shape}" for shape in SHAPES for colour in RGB]
truth = torch.tensor(
    [labels.index(f"a photo of a {colour} {shape}") for shape, colour, _ in scenes]
)
label_embeds = embed_text(labels)


def accuracy(embeddings: torch.Tensor) -> float:
    """Zero-shot accuracy of ``embeddings`` on the twelve colour-and-shape labels."""

    scores = (embeddings / embeddings.norm(dim=-1, keepdim=True)) @ label_embeds.T
    return (scores.argmax(-1) == truth).float().mean().item()


def mean_ablated(names: set[str]) -> torch.Tensor:
    """The embedding with the terms in ``names`` replaced by their mean over pictures."""

    return sum(
        part.mean(0).expand_as(part) if name in names else part for name, part in parts.items()
    )


def is_head(name: str, layers: range) -> bool:
    """Whether ``name`` is an attention head's term in one of ``layers``."""

    return ".H" in name and int(name[1:].split(".")[0]) in layers


groups = {
    "nothing": set(),
    "every MLP": {name for name in parts if name.endswith(".mlp")},
    "heads of layers 0-7": {name for name in parts if is_head(name, range(0, 8))},
    "heads of layers 8-11": {name for name in parts if is_head(name, range(8, 12))},
    "all but heads of layers 8-11": {name for name in parts if not is_head(name, range(8, 12))},
}
scores = {group: accuracy(mean_ablated(names)) for group, names in groups.items()}
for group, score in scores.items():
    print(f"mean-ablate {group:30s} zero-shot accuracy {score:.2f}")
RESULT = (
    abs(scores["every MLP"] - scores["nothing"]) <= 0.05,
    scores["all but heads of layers 8-11"] >= scores["nothing"] - 0.1,
    scores["heads of layers 8-11"] <= scores["nothing"] - 0.3,
)

## 3. TextSpan: describe each late head with sentences

TextSpan takes one head's terms over all pictures (centred), and a pool of sentence embeddings.
It picks the sentence whose direction carries the most of the head's variance, removes that
direction from the head's terms and from every sentence, and repeats. The first few sentences
describe what the head writes. This is the paper's greedy algorithm, without its preliminary
projection onto the head's principal components.

In [ ]:
# recipe-test: tool=clip expected=True
KINDS = {
    "colour": [
        f"the color {c}"
        for c in ("red", "green", "blue", "yellow", "orange", "purple", "black", "white")
    ],
    "shape": [
        f"a {s}" for s in ("circle", "square", "triangle", "star", "ring", "rectangle", "hexagon")
    ],
    "number": ["a single object", "two objects", "three objects", "four objects", "many objects"],
    "other": [f"an object on the {side}" for side in ("left", "right", "top", "bottom")]
    + [
        "a large object",
        "a small object",
        "a striped pattern",
        "a dotted pattern",
        "a smooth surface",
    ]
    + ["a photo of a dog", "a photo of a person", "a landscape", "written text", "an astronaut"]
    + ["a blurry photo", "a drawing", "a bright image", "a dark image", "a plain background"],
}
pool = [sentence for sentences in KINDS.values() for sentence in sentences]
kind_of = {sentence: kind for kind, sentences in KINDS.items() for sentence in sentences}
pool_embeds = embed_text(pool)


def text_span(term: torch.Tensor, steps: int = 3) -> list[str]:
    """Greedy TextSpan: the ``steps`` pool sentences that explain most of ``term``'s variance."""

    residual = term - term.mean(0)
    directions = pool_embeds.clone()
    chosen = []
    for _ in range(steps):
        unit = directions / directions.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        variance = (residual @ unit.T).pow(2).sum(0)
        best = int(variance.argmax())
        chosen.append(pool[best])
        direction = unit[best]
        residual = residual - (residual @ direction)[:, None] * direction
        directions = directions - (directions @ direction)[:, None] * direction
    return chosen


late_heads = [name for name in parts if is_head(name, range(8, 12))]
spread = {name: (parts[name] - parts[name].mean(0)).pow(2).sum().item() for name in late_heads}
busiest = sorted(late_heads, key=spread.get, reverse=True)[:6]
descriptions = {name: text_span(parts[name]) for name in busiest}
for name in busiest:
    print(f"{name:7s} " + "; ".join(descriptions[name]))

first_kind = {name: kind_of[text[0]] for name, text in descriptions.items()}
print(first_kind)
RESULT = {"colour", "shape"} <= set(first_kind.values())

## 4. One head alone carries one property

If a head writes colour, its term alone should tell the colours apart, and tell little else.
Score each late head's centred term against the centred text embeddings of four colour labels,
three shape labels and two number labels.

In [ ]:
# recipe-test: tool=clip expected=True
properties = {
    "colour": (
        [f"a photo of a {c} object" for c in RGB],
        [list(RGB).index(c) for _, c, _ in scenes],
    ),
    "shape": ([f"a photo of a {s}" for s in SHAPES], [SHAPES.index(s) for s, _, _ in scenes]),
    "number": (
        ["a photo of one object", "a photo of three objects"],
        [int(n == 3) for _, _, n in scenes],
    ),
}


def readout(name: str) -> dict[str, float]:
    """Accuracy of one head's centred term on each property."""

    term = parts[name] - parts[name].mean(0)
    result = {}
    for prop, (sentences, answers) in properties.items():
        text = embed_text(sentences)
        text = text - text.mean(0)
        result[prop] = ((term @ text.T).argmax(-1) == torch.tensor(answers)).float().mean().item()
    return result


table = {name: readout(name) for name in busiest}
for name, accuracies in table.items():
    print(f"{name:7s} " + "  ".join(f"{prop} {value:.2f}" for prop, value in accuracies.items()))
colour_head = max(table, key=lambda name: table[name]["colour"] - table[name]["shape"])
shape_head = max(table, key=lambda name: table[name]["shape"] - table[name]["colour"])
print(f"colour head {colour_head}: {descriptions[colour_head]}")
print(f"shape head {shape_head}: {descriptions[shape_head]}")
RESULT = table[colour_head]["colour"] >= 0.85 and table[shape_head]["shape"] >= 0.85

On the pinned checkpoint this prints, among others: layer 11 head 5 reads colour on every
picture but shape on only half of them, and its sentences are colours; layer 11 head 7 reads
shape on 92 percent and colour on 42 percent, and its sentences are shapes. Not every head is
that clean: layer 11 head 3's sentences are colours, yet its term reads shape as well as
colour, and layer 11 head 8's sentences are about number while its term alone cannot tell one
copy from three. TextSpan describes the directions a head's output varies along, which is not
the same as what a linear readout of it can decode; the paper's own heads are labelled the
same way, and checked with retrieval and ablation.

To go further: the paper's pool of about 3,500 sentences and ImageNet pictures, its
preliminary projection onto each head's principal components, and its spatial decomposition,
which splits each head's term over the image patches with the head's `pattern` and `v` facets.